# AeroCast — Phase 2: Evidently Baseline

Builds the **reference dataset** used in Phase 5 drift detection. Generates:
- `data/reference/reference.parquet` — first 30 days as the stable baseline
- `reports/baseline_data_quality.html` — Evidently DataQuality report
- `reports/baseline_drift.html` — Evidently DataDrift report (reference vs. held-out current window)

Run this notebook once after Phase 1 data ingestion. The reference parquet is DVC-tracked; the HTML reports are gitignored (reproducible).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..") / "src"))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from evidently.report import Report
from evidently.metric_preset import DataQualityPreset, DataDriftPreset
from evidently import ColumnMapping

from aerocast.data.client import OpenAQClient
from aerocast.data.preprocess import (
    clean_raw, pivot_to_wide, compute_aqi_column, engineer_features,
)

REFERENCE_PATH = Path("../data/reference")
REPORTS_PATH   = Path("../reports")
REPORTS_PATH.mkdir(exist_ok=True)

print("Environment ready ✓")

## 1. Data Load

Same load strategy as the EDA notebook: DVC-pulled parquet first, live API fallback.
Fetches **60 days** (double the window) so we have enough data to split reference / current.

In [ ]:
PROCESSED_PATH = Path("../data/processed")
parquet_files = list(PROCESSED_PATH.glob("*.parquet"))

if parquet_files:
    print(f"Loading {len(parquet_files)} processed parquet file(s)…")
    raw_df = pd.concat([pd.read_parquet(f) for f in sorted(parquet_files)], ignore_index=True)
    print(f"Loaded {len(raw_df):,} raw rows.")
else:
    print("No processed data found — fetching 60 days live from OpenAQ…")
    import os
    from datetime import datetime, timedelta, timezone
    from dotenv import load_dotenv

    load_dotenv(Path("..") / ".env")
    api_key = os.getenv("OPENAQ_API_KEY", "")
    if not api_key:
        raise EnvironmentError(
            "OPENAQ_API_KEY not set. Add it to .env or export it before running."
        )

    client = OpenAQClient(api_key=api_key)
    locs_df = client.fetch_locations(country="US", limit=20)
    location_ids = locs_df["id"].head(5).tolist()
    print(f"Locations: {location_ids}")

    date_to   = datetime.now(tz=timezone.utc)
    date_from = date_to - timedelta(days=60)

    raw_df = client.fetch_measurements(
        location_ids=location_ids,
        date_from=date_from,
        date_to=date_to,
    )
    print(f"Fetched {len(raw_df):,} raw records.")


## 2. Preprocessing

In [ ]:
cleaned = clean_raw(raw_df)
wide    = pivot_to_wide(cleaned)
wide    = compute_aqi_column(wide)
wide    = engineer_features(wide)
wide    = wide.dropna(subset=["aqi"]).reset_index(drop=True)

print(f"Processed shape: {wide.shape}")
print(f"Date range: {wide['datetime'].min()} → {wide['datetime'].max()}")
wide.head(3)

## 3. Train / Reference Split

The **first 30 days** become the reference dataset; the remainder is the "current" window used to validate the Evidently report.

In [ ]:
wide = wide.sort_values("datetime").reset_index(drop=True)
split_date = wide["datetime"].min() + pd.Timedelta(days=30)

reference = wide[wide["datetime"] <  split_date].copy()
current   = wide[wide["datetime"] >= split_date].copy()

print(f"Reference : {len(reference):,} rows  ({reference['datetime'].min().date()} → {reference['datetime'].max().date()})")
print(f"Current   : {len(current):,} rows  ({current['datetime'].min().date()} → {current['datetime'].max().date()})")

if len(reference) == 0:
    raise ValueError(
        "Reference dataset is empty. Ensure at least 30 days of data are available. "
        "Run the OpenAQ ingestion pipeline or widen the fetch window."
    )

## 4. Save Reference Dataset

Saved to `data/reference/reference.parquet`. This path is DVC-tracked (see `.gitignore` — the directory is excluded from git).

In [ ]:
ref_file = REFERENCE_PATH / "reference.parquet"
reference.to_parquet(ref_file, index=False)
print(f"Saved reference dataset → {ref_file}")
print(f"Shape: {reference.shape}  |  Size: {ref_file.stat().st_size / 1024:.1f} KB")

## 5. Evidently Column Mapping

In [ ]:
pollutant_cols = [c for c in ["pm25", "pm10", "o3", "no2", "so2", "co"] if c in wide.columns]
lag_cols       = [c for c in wide.columns if c.startswith("aqi_lag_")]
roll_cols      = [c for c in wide.columns if c.startswith("aqi_roll_mean_")]
calendar_cols  = [c for c in ["hour", "day_of_week", "month"] if c in wide.columns]

numerical_features = pollutant_cols + lag_cols + roll_cols + calendar_cols
categorical_features = ["is_weekend"] if "is_weekend" in wide.columns else []

# Evidently expects DataFrames without datetime / ID columns
DROP_COLS = ["datetime", "location_id"]

column_mapping = ColumnMapping(
    target="aqi",
    numerical_features=[c for c in numerical_features if c in reference.columns],
    categorical_features=[c for c in categorical_features if c in reference.columns],
)

ref_ev  = reference.drop(columns=DROP_COLS, errors="ignore")
curr_ev = current.drop(columns=DROP_COLS, errors="ignore")

print("Column mapping:")
print(f"  target              : aqi")
print(f"  numerical features  : {len(column_mapping.numerical_features)}")
print(f"  categorical features: {column_mapping.categorical_features}")

## 6. Data Quality Report

Compares basic data-quality metrics (missing values, value ranges, distributions) between the reference and current windows.

In [ ]:
quality_report = Report(metrics=[DataQualityPreset()])
quality_report.run(
    reference_data=ref_ev,
    current_data=curr_ev,
    column_mapping=column_mapping,
)
quality_report.show(mode="inline")

## 7. Data Drift Report

Applies statistical drift tests per column (Wasserstein distance for numerical, chi-squared for categorical). This establishes the **baseline drift expectation** between two adjacent time windows of the same dataset — drift scores here should be low and serve as the "green" reference in Phase 5.

In [ ]:
drift_report = Report(metrics=[DataDriftPreset()])
drift_report.run(
    reference_data=ref_ev,
    current_data=curr_ev,
    column_mapping=column_mapping,
)
drift_report.show(mode="inline")

## 8. Save Reports

In [ ]:
quality_html = REPORTS_PATH / "baseline_data_quality.html"
drift_html   = REPORTS_PATH / "baseline_drift.html"

quality_report.save_html(str(quality_html))
drift_report.save_html(str(drift_html))

print(f"Saved: {quality_html}  ({quality_html.stat().st_size / 1024:.0f} KB)")
print(f"Saved: {drift_html}  ({drift_html.stat().st_size / 1024:.0f} KB)")
print()
print("These HTML files are gitignored — regenerate by re-running this notebook.")

## 9. Drift Summary

In [ ]:
try:
    result  = drift_report.as_dict()["metrics"][0]["result"]
    n_cols  = result.get("number_of_columns", "?")
    n_drift = result.get("number_of_drifted_columns", "?")
    share   = result.get("share_of_drifted_columns", 0)

    print("=" * 50)
    print("Baseline Drift Summary")
    print("=" * 50)
    print(f"Columns assessed : {n_cols}")
    print(f"Drifted columns  : {n_drift}  ({share*100:.1f}%)")
    print()
    if share < 0.2:
        print("✓ Drift share below 20% — reference window is stable.")
        print("  Phase 5 will alert when live data exceeds this baseline.")
    else:
        print("⚠ Higher drift share than expected for adjacent windows.")
        print("  Consider widening the reference window or checking data quality.")
except Exception as e:
    print(f"Could not parse drift result: {e}")
    print("Check the drift report HTML directly.")

print()
print("Reference dataset saved. DVC-track it before moving to Phase 3:")
print()
print("  dvc add data/reference/reference.parquet")
print("  dvc push")